In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import requests, os, sys
from bs4 import BeautifulSoup
from time import sleep
import DataFrameGetter as dfg
import pandas as pd
import numpy as np
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, StaleElementReferenceException

C:\Users\HP\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
#ISL_MAIN_URL = "https://www.scoresway.com/en_GB/soccer/primera-división-2025-2026/80zg2v1cuqcfhphn56u4qpyqc/results"
#ISL_MAIN_URL = "https://www.scoresway.com/en_GB/soccer/serie-a-2025-2026/emdmtfr1v8rey2qru3xzfwges/results"
ISL_MAIN_URL = "https://www.scoresway.com/en_GB/soccer/premier-league-2025-2026/51r6ph2woavlbbpk8f29nynf8/results"
#ISL_MAIN_URL = "https://www.scoresway.com/en_GB/soccer/bundesliga-2025-2026/2bchmrj23l9u42d68ntcekob8/results"
#ISL_MAIN_URL = "https://www.scoresway.com/en_GB/soccer/ligue-1-2025-2026/dbxs75cag7zyip5re0ppsanmc/results"
#ISL_MAIN_URL = "https://www.scoresway.com/en_GB/soccer/primeira-liga-2025-2026/7l4z25o4hc0fmj9rdspcisdn8/results"
#ISL_MAIN_URL = "https://www.scoresway.com/en_GB/soccer/indian-super-league-2025-2026/d70h9cciplfhf01fn96gpcw0k/results"
#ISL_MAIN_URL ="https://www.scoresway.com/en_GB/soccer/fifa-world-cup-2026-canada-mexico-usa/873cbl9cd9butm4air0mugxzo/results"

In [3]:
def getSoup(url):
    res = requests.get(url)
    # FIX Bug 7: specify parser to avoid warning and inconsistent behaviour
    soup = BeautifulSoup(res.text, 'html.parser')
    sleep(3)
    return soup

def getDriver(url):
    driver = webdriver.Chrome()
    driver.get(url)
    driver.implicitly_wait(10)
    sleep(5)
    return driver

def removeCookieDialog(driver):
    # Remove cookie banner and any other common overlays that intercept clicks
    driver.execute_script("""
        var ids = ['onetrust-banner-sdk', 'onetrust-consent-sdk', 'onetrust-pc-sdk'];
        ids.forEach(function(id) {
            var el = document.getElementById(id);
            if (el) el.remove();
        });
        // Also remove any fixed/sticky overlays sitting on top
        document.querySelectorAll(
            '[class*=\"cookie\"], [class*=\"consent\"], [class*=\"gdpr\"], [id*=\"cookie\"]'
        ).forEach(function(el) { el.remove(); });
    """)

def jsClick(driver, element):
    """Always click via JS to bypass any overlay/intercept issues."""
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", element)
    sleep(0.5)
    driver.execute_script("arguments[0].click();", element)

In [4]:
def getMaxElement():
    driver = getDriver(ISL_MAIN_URL)
    removeCookieDialog(driver)
    sleep(1)

    wait = WebDriverWait(driver, 20)

    clicker = wait.until(
        EC.presence_of_element_located((By.CSS_SELECTOR, '.Opta-Exp'))
    )
    # FIX ElementClickInterceptedException: always use jsClick to bypass overlays
    jsClick(driver, clicker)

    ul_element = wait.until(
        EC.presence_of_element_located((By.CSS_SELECTOR, 'ul.Opta-Cf'))
    )
    max_index = len(ul_element.find_elements(By.TAG_NAME, 'li'))

    # FIX Bug 1: use quit() not close() to kill the ChromeDriver process fully
    driver.quit()
    return max_index

In [5]:
max_index = getMaxElement()
print("Total matchweeks:", max_index)

InvalidSessionIdException: Message: invalid session id: session deleted as the browser has closed the connection
from disconnected: not connected to DevTools
  (Session info: chrome=151.0.7922.76); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff65bb78935+14ce5]
	chromedriver!GetHandleVerifier [0x7ff65bb78990+14d40]
	chromedriver!(No symbol) [0x7ff65b6a5afd]
	chromedriver!(No symbol) [0x7ff65b691372]
	chromedriver!(No symbol) [0x7ff65b6b7236]
	chromedriver!(No symbol) [0x7ff65b730bc0]
	chromedriver!(No symbol) [0x7ff65b74dd92]
	chromedriver!(No symbol) [0x7ff65b6f308c]
	chromedriver!(No symbol) [0x7ff65b6f3fb3]
	chromedriver!GetHandleVerifier [0x7ff65c1ae5fb+64a9ab]
	chromedriver!GetHandleVerifier [0x7ff65c1a8b02+644eb2]
	chromedriver!GetHandleVerifier [0x7ff65c1ce39e+66a74e]
	chromedriver!GetHandleVerifier [0x7ff65bb95d6e+3211e]
	chromedriv

In [ ]:
mw_result_links = {}
backup_of_urls = []
failed_matchweeks = []

for i in range(4,5):  # Latest matchweek is index 0
    inner_driver = getDriver(ISL_MAIN_URL)
    removeCookieDialog(inner_driver)
    wait = WebDriverWait(inner_driver, 10)

    try:
        # Remove overlays AGAIN right before clicking — page may re-inject them after load
        sleep(2)
        removeCookieDialog(inner_driver)
        sleep(0.5)
        # Expand the matchweek dropdown — use JS click to bypass any remaining overlay
        clicker = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '.Opta-Exp')))
        jsClick(inner_driver, clicker)

        # Get the dropdown list
        ul_element = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'ul.Opta-Cf')))
        all_mw = ul_element.find_elements(By.TAG_NAME, 'a')

        if len(all_mw) <= i:
            print(i, "Overflow — no more matchweeks")
            # FIX Bug 1: quit() instead of close()
            inner_driver.quit()
            continue

        mw_item = all_mw[i]
        name = mw_item.text.strip() or "Playoff"
        print(i, "WORKING FOR:", name)

        # FIX Bug 6: removed redundant wait.until(EC.element_to_be_clickable(mw_item))
        # (EC.element_to_be_clickable expects a locator tuple, not a WebElement;
        #  we already click via JS below so the wait is unnecessary)
        inner_driver.execute_script("arguments[0].scrollIntoView(true);", mw_item)
        inner_driver.execute_script("arguments[0].click();", mw_item)

        # Wait for fixtures to load
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'tbody.Opta-result.Opta-fixture')))

        urls = []
        # FIX Bug 2: count fixtures first, then re-fetch by index inside the loop
        # to avoid StaleElementReferenceException after navigating back
        num_fixtures = len([
            x for x in inner_driver.find_elements(By.CSS_SELECTOR, 'tbody.Opta-result.Opta-fixture')
            if x.text
        ])

        for j in range(num_fixtures):
            try:
                print(f"  Working on {name}, Game {j+1}/{num_fixtures}")

                # FIX Bug 2: re-fetch fixtures after each back-navigation (DOM is reloaded)
                fixture_tables = [
                    x for x in inner_driver.find_elements(By.CSS_SELECTOR, 'tbody.Opta-result.Opta-fixture')
                    if x.text
                ]
                if j >= len(fixture_tables):
                    print(f"  Game {j+1} index out of range after reload, skipping")
                    continue

                fixture = fixture_tables[j]
                match_page = fixture.find_element(By.CSS_SELECTOR, '[title="View match"]')
                inner_driver.execute_script("arguments[0].scrollIntoView(true);", match_page)
                inner_driver.execute_script("arguments[0].click();", match_page)

                removeCookieDialog(inner_driver)

                # Collect Player Stats URL
                striplist = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'ul.striplist')))
                result_urls = [
                    x.find_element(By.TAG_NAME, 'a').get_property('href')
                    for x in striplist.find_elements(By.TAG_NAME, 'li')
                    if x.text.strip() == 'PLAYER STATS'
                ]
                if result_urls:
                    urls.append((j, result_urls[0]))
                    print(f"  Collected: {result_urls[0]}")
                else:
                    print(f"  No PLAYER STATS link found for game {j+1}")

                # Go back and wait for fixtures table to reload
                inner_driver.execute_script("window.history.go(-1)")
                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'tbody.Opta-result.Opta-fixture')))

            # FIX Bug 3: continue instead of break so one failing game
            # doesn't abandon the rest of the matchweek
            except StaleElementReferenceException as e:
                print(f"  Stale element on {name}, Game {j+1} — retrying next game: {e}")
                inner_driver.execute_script("window.history.go(-1)")
                sleep(2)
                continue
            except Exception as e:
                print(f"  Error on {name}, Game {j+1}: {type(e).__name__}: {e}")
                try:
                    inner_driver.execute_script("window.history.go(-1)")
                    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'tbody.Opta-result.Opta-fixture')))
                except Exception:
                    pass
                continue  # skip this game, move to next

        if name not in mw_result_links:
            mw_result_links[name] = urls

        backup_of_urls.append(urls)
        print(f"  Done {name}: {len(urls)} URLs collected")

    except Exception as e:
        print(f"Fatal error for matchweek index {i}: {type(e).__name__}: {e}")
        failed_matchweeks.append(i)
    finally:
        # FIX Bug 1: always quit() to avoid orphan ChromeDriver processes
        inner_driver.quit()

print("\nDone. Failed matchweek indices:", failed_matchweeks)

In [7]:
# FIX Bug 4: removed the redundant first cell that used the side-effect extend pattern.
# This single cell is the correct, clean version.
all_urls = [y for x in backup_of_urls for y in x]
print(f"Total URLs collected: {len(all_urls)}")

Total URLs collected: 1


In [6]:
all_urls =[(0,"https://www.scoresway.com/en_GB/soccer/premier-league-2025-2026/51r6ph2woavlbbpk8f29nynf8/match/view/56sqj49wua0241sbk71k9rdhw/player-stats")]
          

In [7]:
len(all_urls)

1

In [8]:
f_U_ = []
for entry in all_urls:
    # Defensive unpack — each entry is (fixture_index, url)
    try:
        _, url = entry
    except (TypeError, ValueError):
        print(f"Skipping malformed entry in all_urls: {entry!r}")
        continue

    try:
        response = dfg.fetch_event_response(url)
        match_info = response['matchevent']['response_content']['matchInfo']
        home = [x['officialName'].replace(' ', '_') for x in match_info['contestant'] if x.get('position') == 'home'][0]
        away = [x['officialName'].replace(' ', '_') for x in match_info['contestant'] if x.get('position') == 'away'][0]
        file_name = f"{match_info['localDate'].replace('-', '_')}_{home}_vs_{away}.xlsx"
        full_file_path = rf"D:\Leagues Data\League data\Scoresway\2025_26_Season\EPL_scoresway_2025_26\{file_name}"

        if os.path.exists(full_file_path):
            print(f"Already exists, skipping: {file_name}")
            continue

        element = response['matchevent']
        content_element = element['response_content']

        # Get team names
        teams_df = pd.DataFrame(content_element['matchInfo']['contestant'])

        # Get Opta event type IDs
        type_id_df = pd.read_csv(r'./opta_data/template/opta-events.csv', encoding='ISO-8859-1')
        type_id_df.rename(columns={'ID': 'typeId'}, inplace=True)

        # Get Opta qualifier IDs
        qualifiers_df = pd.read_csv(r'./opta_data/template/opta-qualifiers.csv')

        # Build events DataFrame
        events_df = pd.DataFrame(content_element['liveData']['event'])

        id1 = teams_df['id'].iloc[0]
        team1 = teams_df['officialName'].iloc[0]
        id2 = teams_df['id'].iloc[1]
        team2 = teams_df['officialName'].iloc[1]

        events_df['contestantId'] = np.where(events_df['contestantId'] == id1, team1, team2)
        events_df['outcome'] = np.where(events_df['outcome'] == 1, 'Successful', 'Unsuccessful')
        events_df.rename(columns={'contestantId': 'Team', 'timeMin': 'Minute', 'timeSec': 'Seconds'}, inplace=True)

        event_action_result = events_df.apply(dfg.return_event_data_and_description, args=(type_id_df,), axis=1)
        events_df['typeId'], events_df['Description'] = zip(*event_action_result)
        events_df['qualifier'] = events_df.apply(dfg.return_qualifier_data, args=(qualifiers_df,), axis=1)

        events_df.to_excel(full_file_path, index=False)
        print(f"Saved: {file_name}")

    except Exception as e:
        print(f"Error for URL {url}: {e}")
        f_U_.append(url)

print(f"\nFailed URLs ({len(f_U_)}):")
for u in f_U_:
    print(" ", u)

Saved: 2026_05_24_Crystal_Palace_FC_vs_Arsenal_FC.xlsx

Failed URLs (0):
